# Lift, Splat, Shoot: a standalone walkthrough
This notebook defines the current **practical** segmentation architecture locally. It uses synthetic CPU data, random initialization, and no network or dataset by default. It is not a reproduced paper result or a verified original-checkpoint implementation. The optional sections use this repository's data, model, or evaluation infrastructure and are labeled where used.

```text
images [B,N,3,H,W] ── shared camera encoder ──┬── depth probabilities [B,N,D,h,w]
                                             └── context [B,N,C,h,w]
              calibration + pixel/depth frustum ─── outer product ── voxel sum
                                                        │
                                          BEV decoder ──┴── logits [B,1,X,Y]
```
The model learns a probability distribution over depth rather than requiring depth labels. Its voxel grid is indexed by ego-frame x and y; visualization below labels these axes explicitly. See `docs/paper-to-code.md` for reference deviations and evidence.


In [ ]:
import math
import torch
from torch import nn
from torch.nn import functional as F
from torchvision.models import efficientnet_b0, resnet18, EfficientNet_B0_Weights
torch.manual_seed(0)
torch.set_num_threads(1)


## 1. Camera encoder and lift
**Intuition:** each image location can contribute to multiple 3-D positions, weighted by learned depth probabilities. With logits `a`, context `c`, and depth index `d`, `p(d)=exp(a_d)/sum_j exp(a_j)` and `lift(d,c)=p(d)c`. Images are `[B,N,3,H,W]`, depth is `[B,N,D,h,w]`, and context is `[B,N,C,h,w]`. A stride-16 skip and stride-32 features are fused.
The probe uses random initialization, evaluation mode, and one image; no pretrained weights are fetched. `pretrained=True` in the class may download TorchVision weights and is deliberately not used by this notebook's default run.


In [ ]:
class CameraEncoder(nn.Module):
    def __init__(self, depth_bins, context_channels, pretrained=False):
        super().__init__()
        if depth_bins <= 0 or context_channels <= 0: raise ValueError('channels must be positive')
        weights = EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        self.features = efficientnet_b0(weights=weights).features[:8]
        self.depth_bins, self.context_channels = depth_bins, context_channels
        self.fuse = nn.Sequential(nn.Conv2d(432, 512, 3, padding=1, bias=False), nn.BatchNorm2d(512), nn.ReLU(inplace=True), nn.Conv2d(512, 512, 3, padding=1, bias=False), nn.BatchNorm2d(512), nn.ReLU(inplace=True))
        self.projection = nn.Conv2d(512, depth_bins + context_channels, 1)
    def forward(self, images):
        if images.ndim != 5 or images.shape[2] != 3: raise ValueError('images must have shape [B,N,3,H,W]')
        batch, cameras, channels, height, width = images.shape
        if height < 32 or width < 32 or height % 32 or width % 32: raise ValueError('image dimensions must be divisible by 32')
        x = images.reshape(batch * cameras, channels, height, width)
        for stage in self.features[:6]: x = stage(x)
        skip = x
        for stage in self.features[6:8]: x = stage(x)
        x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=True)
        x = self.projection(self.fuse(torch.cat((skip, x), dim=1)))
        depth = x[:, :self.depth_bins].softmax(dim=1)
        context = x[:, self.depth_bins:]
        return depth.reshape(batch, cameras, self.depth_bins, *depth.shape[-2:]), context.reshape(batch, cameras, self.context_channels, *context.shape[-2:])


In [ ]:
camera_probe = CameraEncoder(2, 8).eval()
with torch.no_grad():
    probe_depth, probe_context = camera_probe(torch.zeros(1, 1, 3, 32, 64))
assert probe_depth.shape == (1, 1, 2, 2, 4) and probe_context.shape == (1, 1, 8, 2, 4)
assert torch.allclose(probe_depth.sum(dim=2), torch.ones(1, 1, 2, 4))


## 2. Geometry
**Intuition:** each pixel and depth gives a point along a camera ray. Frustum points `(u,v,d)` are sampled in input-image pixels. Undo image augmentation first, then compute `p_camera = K⁻¹ [ud,vd,d]` and `p_ego = R p_camera + t`. The tensor is `[B,N,D,h,w,3]`; coordinates after unprojection are in metres. `K` is the camera intrinsic matrix and `(R,t)` is camera-to-ego calibration.
The raw-data loader uses timestamp-compensated camera-to-LiDAR-ego transforms, a documented difference from the pinned reference. The analytical probe uses identity calibration so the expected result can be calculated by hand.


In [ ]:
def make_frustum(height, width, depths, device=None, image_size=None):
    if height <= 0 or width <= 0: raise ValueError('frustum dimensions must be positive')
    image_height, image_width = image_size if image_size is not None else (height, width)
    if image_height <= 0 or image_width <= 0: raise ValueError('image dimensions must be positive')
    d = torch.as_tensor(depths, device=device)
    if not d.is_floating_point(): d = d.to(dtype=torch.float32)
    y, x = torch.meshgrid(torch.linspace(0, image_height - 1, height, dtype=d.dtype, device=d.device), torch.linspace(0, image_width - 1, width, dtype=d.dtype, device=d.device), indexing='ij')
    return torch.stack((x[None].expand(len(d), -1, -1), y[None].expand(len(d), -1, -1), d[:, None, None].expand(-1, height, width)), -1)

def camera_to_ego(points, intrinsics, extrinsics, post_rots=None, post_trans=None):
    if points.ndim != 6 or intrinsics.shape[:2] != points.shape[:2] or extrinsics.shape[:2] != points.shape[:2]: raise ValueError('batch and camera axes must match')
    b, c, d, h, w, _ = points.shape
    pix = points.reshape(b, c, -1, 3)
    if (post_rots is None) != (post_trans is None): raise ValueError('post transforms must be provided together')
    if post_rots is not None:
        if post_rots.shape != (b, c, 3, 3) or post_trans.shape != (b, c, 3): raise ValueError('invalid post-transform shape')
        pix = torch.linalg.solve(post_rots, (pix - post_trans.unsqueeze(2)).transpose(-1, -2)).transpose(-1, -2)
    xyz = torch.linalg.solve(intrinsics, torch.cat((pix[..., :2] * pix[..., 2:3], pix[..., 2:3]), -1).transpose(-1, -2)).transpose(-1, -2)
    return ((extrinsics[..., :3, :3].unsqueeze(2) @ xyz.unsqueeze(-1)).squeeze(-1) + extrinsics[..., :3, 3].unsqueeze(2)).reshape(b, c, d, h, w, 3)


In [ ]:
frustum_probe = make_frustum(2, 2, [2.0], image_size=(8, 8))[None, None]
geometry_probe = camera_to_ego(frustum_probe, torch.eye(3)[None, None], torch.eye(4)[None, None])
assert torch.allclose(geometry_probe[0, 0, 0, -1, -1], torch.tensor([14., 14., 2.]))


## 3. Splat and BEV decoder
**Intuition:** aggregate evidence from all cameras into a common top-down grid. For cell `v`, `BEV[v] = sum_i 1[p_i in v] * lift_i`. The validity mask excludes points outside the half-open bounds `[lower, upper)` before floor-based binning; it is a geometric mask, not a learned confidence. The output is `[B,Z×C,X,Y]`, with height bins concatenated in z-major order. A ResNet-block BEV decoder returns logits on the original grid.
Gradients flow from the loss to the summed features and hence depth/context predictions. The discrete cell assignment itself is not differentiable with respect to point coordinates. The probe verifies that two points in one cell sum to five.


In [ ]:
class VoxelPool(nn.Module):
    def __init__(self, x_bounds, y_bounds, z_bounds):
        super().__init__()
        bounds = (x_bounds, y_bounds, z_bounds); sizes = []
        for lower, upper, step in bounds:
            if not all(math.isfinite(v) for v in (lower, upper, step)) or step <= 0 or upper <= lower: raise ValueError('invalid voxel bounds')
            count = (upper - lower) / step
            if round(count) < 1 or not math.isclose(count, round(count), rel_tol=1e-7, abs_tol=1e-7): raise ValueError('voxel bounds must define an integral number of bins')
            sizes.append(int(round(count)))
        self.grid_size = tuple(sizes)
        self.bounds = tuple(tuple(float(v) for v in bound) for bound in bounds)
    def get_extra_state(self): return {'bounds': self.bounds}
    def set_extra_state(self, state):
        if state.get('bounds') != self.bounds: raise RuntimeError('checkpoint voxel bounds do not match')
    def forward(self, points, features):
        if points.ndim != 6 or points.shape[-1] != 3 or features.ndim != 6: raise ValueError('invalid shape')
        if not points.is_floating_point() or not features.is_floating_point(): raise TypeError('floating point required')
        if points.shape[:-1] != features.shape[:-1] or points.device != features.device: raise ValueError('sample dimensions and device must match')
        batch, channels = points.shape[0], features.shape[-1]; nx, ny, nz = self.grid_size
        lower = points.new_tensor([v[0] for v in self.bounds]); upper = points.new_tensor([v[1] for v in self.bounds]); step = points.new_tensor([v[2] for v in self.bounds])
        valid = ((points >= lower) & (points < upper)).all(dim=-1)
        xyz = torch.floor((points[valid] - lower) / step).long()
        xyz = torch.minimum(xyz.clamp_min(0), torch.tensor((nx-1, ny-1, nz-1), device=points.device))
        batch_index = torch.arange(batch, device=points.device).view(batch, 1, 1, 1, 1).expand(points.shape[:-1])[valid]
        index = ((batch_index * nz + xyz[:, 2]) * nx + xyz[:, 0]) * ny + xyz[:, 1]
        flat = features.new_zeros((batch * nz * nx * ny, channels)).index_add(0, index, features[valid])
        return flat.reshape(batch, nz, nx, ny, channels).permute(0, 1, 4, 2, 3).reshape(batch, nz * channels, nx, ny)

class BEVEncoder(nn.Module):
    def __init__(self, in_channels, out_channels=1):
        super().__init__()
        if in_channels <= 0 or out_channels <= 0: raise ValueError('channels must be positive')
        self.in_channels = in_channels; trunk = resnet18(weights=None, zero_init_residual=True)
        self.stem = nn.Sequential(nn.Conv2d(in_channels, 64, 7, stride=2, padding=3, bias=False), trunk.bn1, trunk.relu)
        self.layer1, self.layer2, self.layer3 = trunk.layer1, trunk.layer2, trunk.layer3
        self.fuse = nn.Sequential(nn.Conv2d(320, 256, 3, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.Conv2d(256, 256, 3, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU(inplace=True))
        self.output = nn.Sequential(nn.Conv2d(256, 128, 3, padding=1, bias=False), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.Conv2d(128, out_channels, 1))
    def forward(self, bev):
        if bev.ndim != 4 or bev.shape[1] != self.in_channels: raise ValueError('invalid BEV features')
        height, width = bev.shape[-2:]
        if height < 8 or width < 8 or height % 8 or width % 8: raise ValueError('BEV dimensions must be divisible by 8')
        skip = self.layer1(self.stem(bev)); x = self.layer3(self.layer2(skip))
        x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=True)
        x = self.fuse(torch.cat((skip, x), dim=1))
        x = F.interpolate(x, size=(height, width), mode='bilinear', align_corners=True)
        return self.output(x)


In [ ]:
pool_probe = VoxelPool((0, 2, 1), (0, 1, 1), (0, 1, 1))
point_probe = torch.tensor([[[[[[0.2, 0.2, 0.2], [0.3, 0.2, 0.2]]]]]])
feature_probe = torch.tensor([[[[[[2.], [3.]]]]]])
assert pool_probe(point_probe, feature_probe)[0, 0, 0, 0] == 5


## 4. Complete segmentation model
The model combines the camera encoder, calibrated projection, sum pooling and BEV decoder. Defaults are source-oriented; the tiny probe below is a synthetic diagnostic configuration. Pretrained weights and checkpoint parity are not assumed.


In [ ]:
class LSSSegmentation(nn.Module):
    def __init__(self, x_bounds=(-50., 50., 0.5), y_bounds=(-50., 50., 0.5), z_bounds=(-10., 10., 20.), depth_bounds=(4., 45., 1.), context_channels=64, out_channels=1, pretrained=False):
        super().__init__()
        start, stop, step = depth_bounds
        if not all(math.isfinite(v) for v in depth_bounds) or start <= 0 or stop <= start or step <= 0: raise ValueError('invalid depth bounds')
        depth_values = torch.arange(start, stop, step, dtype=torch.float32)
        if depth_values.numel() == 0: raise ValueError('empty depth bins')
        self.register_buffer('depth_values', depth_values)
        self.camera_encoder = CameraEncoder(depth_values.numel(), context_channels, pretrained=pretrained)
        self.voxel_pool = VoxelPool(x_bounds, y_bounds, z_bounds)
        self.bev_encoder = BEVEncoder(context_channels * self.voxel_pool.grid_size[2], out_channels)
    def forward(self, images, intrinsics, extrinsics, post_rots=None, post_trans=None):
        depth, context = self.camera_encoder(images)
        frustum = make_frustum(depth.shape[-2], depth.shape[-1], self.depth_values, device=images.device, image_size=images.shape[-2:])
        points = frustum[None, None].expand(images.shape[0], images.shape[1], -1, -1, -1, -1)
        points = camera_to_ego(points, intrinsics, extrinsics, post_rots=post_rots, post_trans=post_trans)
        lifted = depth.unsqueeze(-1) * context.permute(0, 1, 3, 4, 2).unsqueeze(2)
        return self.bev_encoder(self.voxel_pool(points, lifted))


## 5. Sample preparation, objective, and optimization
The sample below has two synthetic cameras, a 16×16 occupancy target and identity camera-to-ego transforms. It is a shape/gradient diagnostic, not a realistic driving scene. Real data additionally requires image normalization, calibrated cameras and rasterized annotations.
For logits `z` and binary labels `y`, the objective is `BCEWithLogits(z,y) = -y log(sigmoid(z)) - (1-y) log(1-sigmoid(z))`. The paper and reference use different positive weights; this probe uses weight 1. The optimizer performs a real Adam update. The printed IoU is calculated on pre-update synthetic logits only, not a validation result.


In [ ]:
model = LSSSegmentation(x_bounds=(-8, 8, 1), y_bounds=(-8, 8, 1), z_bounds=(0, 3, 3), depth_bounds=(1, 3, 1), context_channels=8)
images = torch.randn(1, 2, 3, 32, 64)
k = torch.tensor([[10., 0., 32.], [0., 10., 16.], [0., 0., 1.]])[None, None].expand(1, 2, 3, 3).clone()
extrinsics = torch.eye(4)[None, None].expand(1, 2, 4, 4).clone()
target = torch.zeros(1, 1, 16, 16)
target[:, :, 5:11, 6:10] = 1
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
before = model.bev_encoder.output[-1].weight.detach().clone()
optimizer.zero_grad()
logits = model(images, k, extrinsics)
loss = F.binary_cross_entropy_with_logits(logits, target)
loss.backward(); optimizer.step()
assert torch.isfinite(loss) and not torch.equal(before, model.bev_encoder.output[-1].weight)
pred = logits.detach() > 0; truth = target.bool()
intersection = (pred & truth).sum().item(); union = (pred | truth).sum().item()
print('synthetic_loss', float(loss.detach()), 'synthetic_iou', intersection / union if union else None)


## 6. Inference, metric, and visualization
Switch to evaluation mode before inference: BatchNorm uses its running statistics and gradients are unnecessary. Decode occupancy by thresholding logits at zero (equivalent to probability 0.5). IoU is `intersection / union`; an empty union is undefined here. The plot is deliberately labeled synthetic and uses x as the first BEV tensor dimension; it does not depict model performance on nuScenes.


In [ ]:
import matplotlib.pyplot as plt
model.eval()
with torch.no_grad():
    inference_logits = model(images, k, extrinsics)
    inference_probability = inference_logits.sigmoid()
    inference_mask = inference_logits > 0
assert inference_probability.shape == target.shape and torch.isfinite(inference_probability).all()
fig, axes = plt.subplots(1, 3, figsize=(9, 3), constrained_layout=True)
for ax, data, title in zip(axes, (target[0, 0], inference_probability[0, 0], inference_mask[0, 0]), ('synthetic target', 'synthetic probability', 'synthetic prediction')):
    ax.imshow(data.numpy().T, origin='lower', extent=(-8, 8, -8, 8), vmin=0, vmax=1)
    ax.set(title=title, xlabel='ego x (m)', ylabel='ego y (m)')
plt.show()
plt.close(fig)


## 7. Shoot: experimental trajectory scoring
A separate cost-map model scores each candidate trajectory. For a cost map `C` and template cells `q[k,t]`, `logit[k] = -sum_t C[q[k,t]]`; `softmax(logit)` favors lower cost. Metric trajectories are `[B,K,T,2]`, cost maps `[B,X,Y]`, and probabilities `[B,K]`. All points must be inside the same half-open grid; there is no trajectory validity mask or interpolation. Gradients flow to sampled costs, not discrete indices.
This is the repository's **experimental model primitive**, not a complete planning dataset, training protocol, or validated planning metric. The synthetic probe below checks scoring and gradients; it does not claim a planning result.


In [ ]:
def metric_trajectory_cells(trajectories, x_bounds, y_bounds):
    if trajectories.ndim != 4 or trajectories.shape[-1] != 2 or any(size == 0 for size in trajectories.shape[:-1]): raise ValueError('trajectories must have nonempty shape [B,K,T,2]')
    if not trajectories.is_floating_point() or not torch.isfinite(trajectories).all(): raise ValueError('trajectory coordinates must be finite floating point values')
    sizes = []
    for lower, upper, step in (x_bounds, y_bounds):
        if not all(math.isfinite(value) for value in (lower, upper, step)) or step <= 0 or upper <= lower: raise ValueError('grid bounds must be finite and increasing')
        count = (upper - lower) / step
        if round(count) < 1 or not math.isclose(count, round(count), rel_tol=1e-7, abs_tol=1e-7): raise ValueError('grid bounds must define an integral number of bins')
        sizes.append(int(round(count)))
    lower = trajectories.new_tensor([x_bounds[0], y_bounds[0]])
    upper = trajectories.new_tensor([x_bounds[1], y_bounds[1]])
    step = trajectories.new_tensor([x_bounds[2], y_bounds[2]])
    if not ((trajectories >= lower) & (trajectories < upper)).all(): raise ValueError('trajectory coordinates are outside the BEV grid')
    cells = torch.floor((trajectories - lower) / step).long()
    max_index = torch.tensor([sizes[0] - 1, sizes[1] - 1], device=trajectories.device)
    return torch.minimum(cells.clamp_min(0), max_index)

def trajectory_logits(cost_maps, cells):
    if cost_maps.ndim != 3 or cells.ndim != 4 or cells.shape[-1] != 2: raise ValueError('expected cost maps [B,X,Y] and cells [B,K,T,2]')
    if cost_maps.shape[0] != cells.shape[0] or any(size == 0 for size in cells.shape[:3]): raise ValueError('trajectory batches and dimensions must be nonempty and aligned')
    if not cost_maps.is_floating_point(): raise TypeError('cost maps must be floating point')
    if cells.dtype not in (torch.int32, torch.int64): raise TypeError('trajectory cells must use integer indices')
    if cells.device != cost_maps.device: raise ValueError('trajectory cells and cost maps must share a device')
    x, y = cells[..., 0], cells[..., 1]
    if ((x < 0) | (x >= cost_maps.shape[1]) | (y < 0) | (y >= cost_maps.shape[2])).any(): raise ValueError('trajectory cell is outside the cost map')
    batch = torch.arange(cost_maps.shape[0], device=cost_maps.device)[:, None, None]
    return -cost_maps[batch, x.long(), y.long()].sum(dim=-1)

class LSSPlanner(nn.Module):
    def __init__(self, x_bounds=(-50., 50., .5), y_bounds=(-50., 50., .5), **model_kwargs):
        super().__init__()
        self.cost_model = LSSSegmentation(x_bounds=x_bounds, y_bounds=y_bounds, out_channels=1, **model_kwargs)
    def forward(self, images, intrinsics, extrinsics, trajectories, post_rots=None, post_trans=None):
        cost = self.cost_model(images, intrinsics, extrinsics, post_rots=post_rots, post_trans=post_trans)[:, 0]
        bounds = self.cost_model.voxel_pool.bounds
        cells = metric_trajectory_cells(trajectories, bounds[0], bounds[1])
        logits = trajectory_logits(cost, cells)
        return {'cost_map': cost, 'logits': logits, 'probabilities': logits.softmax(dim=1)}


In [ ]:
cost_probe = torch.tensor([[[0., 1.], [2., 3.]]], requires_grad=True)
trajectory_probe = torch.tensor([[[[0.1, 0.1], [0.1, 1.1]], [[1.1, 0.1], [1.1, 1.1]]]])
cell_probe = metric_trajectory_cells(trajectory_probe, (0, 2, 1), (0, 2, 1))
score_probe = trajectory_logits(cost_probe, cell_probe)
torch.testing.assert_close(score_probe, torch.tensor([[-1., -5.]]))
score_probe.sum().backward()
assert torch.count_nonzero(cost_probe.grad) == 4
print('synthetic trajectory probabilities', score_probe.detach().softmax(dim=1))


## 8. Explicit opt-ins: data, checkpoints, parity, and evaluation
Run from the repository root or `notebooks/` directory with the package installed for infrastructure imports. The default remains offline and synthetic. Set environment variables before starting the kernel: `LSS_NOTEBOOK_REAL_DATA=1` and `LSS_NOTEBOOK_DATAROOT=/path/to/nuscenes` load one validation sample; `LSS_NOTEBOOK_CHECKPOINT=1` and `LSS_NOTEBOOK_CHECKPOINT_PATH=/path/to/checkpoint.pt` load a compatible practical epoch checkpoint; `LSS_NOTEBOOK_PARITY=1` compares this notebook model to the installed package on the synthetic sample. For a mini dataset set `LSS_NOTEBOOK_VERSION=v1.0-mini`. These optional paths import repository infrastructure, not the notebook architecture.
`LSS_NOTEBOOK_EVALUATE=1` runs the repository's practical full validation CLI with the supplied checkpoint and dataroot, using the version stored in the checkpoint. This can be slow and is not an official paper metric. It requires both paths and does not train. Missing requested assets fail instead of silently returning synthetic results. Use trusted checkpoints only.


In [ ]:
import os
import json
import importlib
import subprocess
import sys
from pathlib import Path
run_real = os.environ.get('LSS_NOTEBOOK_REAL_DATA') == '1'
run_checkpoint = os.environ.get('LSS_NOTEBOOK_CHECKPOINT') == '1'
run_parity = os.environ.get('LSS_NOTEBOOK_PARITY') == '1'
run_evaluation = os.environ.get('LSS_NOTEBOOK_EVALUATE') == '1'
data_root = Path(os.environ.get('LSS_NOTEBOOK_DATAROOT', '')).expanduser().resolve()
checkpoint_path = Path(os.environ.get('LSS_NOTEBOOK_CHECKPOINT_PATH', '')).expanduser().resolve()
version = os.environ.get('LSS_NOTEBOOK_VERSION', 'v1.0-trainval')
if run_checkpoint or run_evaluation:
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f'checkpoint file not found: {checkpoint_path}')
    saved = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    if not isinstance(saved, dict) or saved.get('format_version') != 1 or saved.get('types', {}).get('model') != 'lss.models.segmentation.LSSSegmentation':
        raise ValueError('unsupported practical segmentation checkpoint')
    config = json.loads(saved['config_json'])
    if run_evaluation:
        version = config['version']
if run_real or run_evaluation:
    if not (data_root / version).is_dir():
        raise FileNotFoundError(f'nuScenes version directory not found: {data_root / version}')
if run_real:
    Dataset = importlib.import_module('lss.data.nuscenes_dataset').NuScenesCameraDataset
    real_dataset = Dataset(dataroot=data_root, version=version, split='val', image_size=(128, 352), bev_bounds=((-50., 50., .5), (-50., 50., .5)))
    real_sample = real_dataset[0]
    print('real validation sample:', {key: tuple(real_sample[key].shape) for key in ('images', 'intrinsics', 'extrinsics', 'bev_target')})
if run_checkpoint:
    checkpoint_model = LSSSegmentation(x_bounds=config['x_bounds'], y_bounds=config['y_bounds'], z_bounds=config['z_bounds'], depth_bounds=config['depth_bounds'], context_channels=config['context_channels'], out_channels=config['out_channels'], pretrained=False)
    checkpoint_model.load_state_dict(saved['model'], strict=True)
    checkpoint_model.eval()
    print('loaded practical checkpoint epoch', saved['epoch'], 'recipe', config['recipe'])
if run_parity:
    PackageModel = importlib.import_module('lss.models.segmentation').LSSSegmentation
    package_model = PackageModel(x_bounds=(-8, 8, 1), y_bounds=(-8, 8, 1), z_bounds=(0, 3, 3), depth_bounds=(1, 3, 1), context_channels=8).eval()
    package_model.load_state_dict(model.state_dict(), strict=True)
    with torch.no_grad():
        package_logits = package_model(images, k, extrinsics)
    torch.testing.assert_close(inference_logits, package_logits, rtol=1e-5, atol=1e-6)
    print('synthetic notebook/package output parity passed')
if run_evaluation:
    repo_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'eval.py').is_file() and (p / 'lss').is_dir()), None)
    if repo_root is None:
        raise FileNotFoundError('run evaluation from the repository root or its notebooks directory')
    command = [sys.executable, str(repo_root / 'eval.py'), '--checkpoint', str(checkpoint_path), '--dataroot', str(data_root), '--device', 'cpu']
    subprocess.run(command, cwd=repo_root, check=True)
print('LSS_NOTEBOOK_COMPLETE')
